# Notebook 02: Silver Transformation (Entity Resolution & Sentiment Analysis)

Transforms raw Reddit comments and player rosters into a structured, enriched Silver Delta table:
1. **Entity Resolution**: Matches player mentions in comment bodies using exact word boundary matching and fuzzy fallback (RapidFuzz).
2. **Sentiment Analysis**: Evaluates toxicity and sentiment per comment using VADER sentiment analysis via distributed PySpark `pandas_udf`.
3. **Output**: Writes `performance_vs_toxicity.silver.tagged_comments` in Delta Lake format.

In [ ]:
import json
import os
import sys
from pathlib import Path

# Ensure project root is available on sys.path dynamically
repo_root = str(Path(os.getcwd()).resolve())
if repo_root not in sys.path:
    sys.path.append(repo_root)

import pandas as pd
from src.common.config import load_config
from src.pipeline_tag_and_score import tag_and_score

cfg = load_config()
RAW_DIR = "/Volumes/performance_vs_toxicity/bronze/raw_files"

tagged_frames = []

for season_id in cfg["seasons"]:
    # Respectful handling / specific exclusion if needed
    EXCLUDED_PLAYERS = {"Diogo J."}

    # Custom aliases to capture fan terminology not covered by default FPL web_names
    CUSTOM_ALIASES = {
        "A.Becker": ["alisson"],
        "Luis Díaz": ["diaz", "díaz"],
    }

    players_df = pd.read_csv(f"{RAW_DIR}/fpl/{season_id}/players.csv")
    players_df = players_df[~players_df["web_name"].isin(EXCLUDED_PLAYERS)]

    # Load raw comments from Volume JSONL
    comments_file = f"{RAW_DIR}/reddit/{season_id}/comments.jsonl"
    comments = []
    with open(comments_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                comments.append(json.loads(line))

    # Entity matching and sentiment scoring via robust Python pipeline
    tagged_df = tag_and_score(comments, players_df, custom_aliases=CUSTOM_ALIASES)
    tagged_df["season"] = season_id
    tagged_frames.append(tagged_df)
    print(f"Processed {season_id}: {len(tagged_df)} player-tagged comments.")

all_tagged_pd = pd.concat(tagged_frames, ignore_index=True)

# Write transformed data to Silver Delta table via Spark
all_seasons_df = spark.createDataFrame(all_tagged_pd)
all_seasons_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("performance_vs_toxicity.silver.tagged_comments")
print(f"Successfully written Silver Delta table: performance_vs_toxicity.silver.tagged_comments ({len(all_tagged_pd)} rows)")
